In [ ]:
import base64

In [ ]:
import re

In [ ]:
import requests

In [ ]:
import json

In [ ]:
import sys

In [ ]:
import io

In [ ]:
from PIL import Image

In [ ]:
from IPython.display import Image as IPImage, display

In [ ]:
from pdf2image import convert_from_path

In [ ]:
import os

In [ ]:
from pypdf import PdfReader

In [ ]:
print(Image)

In [ ]:
def resize_convert_image_to_base64(image_path, max_size=1024):
    """
    Convert an image file to a base64 encoded string.
    
    :param image_path: Path to the image file.
    :return: Base64 encoded string of the image.
    """
    if not isinstance(image_path, str):
        raise ValueError("Image path must be a string.")

    ext = os.path.splitext(image_path)[1].lower()

    if ext == '.pdf':
        images = convert_from_path(image_path, dpi=300)
        img = images[0]
    else:
        img = Image.open(image_path)

    
    img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
    print(img.size)

    if img.mode in ('RGBA', 'P'):
        img = img.convert('RGB')

    buffer = io.BytesIO()
    img.save(buffer, format='JPEG')
    buffer.seek(0)
    image_base64 = base64.b64encode(buffer.read()).decode('utf-8')
    
    return image_base64


In [ ]:
def get_ollama32_vision_response(
        image_base64: str,
        prompt: str,
        model: str = "llama3.2-vision"
    ):
    """
    Get a response from the Ollama32 Vision API.
    
    :return: Response from the API.
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "images": [image_base64],
        "stream": False
    }
    
    url = "http://localhost:11434/api/generate"
    
    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()  # Raise an error for bad responses
        data = response.json()
        return data.get('response', 'No response found in the API output.').strip()
    except requests.RequestException as e:
        print(f"An error occurred while making the request: {e}", file=sys.stderr)
        return None

In [ ]:
def get_ollama32_text_response(
        prompt: str,
        model: str = "llama3.2"
    ):
    """
    Get a response from the Ollama32 Text API.
    
    :return: Response from the API.
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "json": True
    }
    
    url = "http://localhost:11434/api/generate"
    
    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()  # Raise an error for bad responses
        data = response.json()
        return data.get('response', 'No response found in the API output.').strip()
    except requests.RequestException as e:
        print(f"An error occurred while making the request: {e}", file=sys.stderr)
        return None

In [ ]:
def get_ollama31_text_response(
        prompt: str,
        model: str = "llama3.1"
    ):
    """
    Get a response from the Ollama31 Text API.
    
    :return: Response from the API.
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "json": True
    }
    
    url = "http://localhost:11434/api/generate"
    
    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()  # Raise an error for bad responses
        data = response.json()
        return data.get('response', 'No response found in the API output.').strip()
    except requests.RequestException as e:
        print(f"An error occurred while making the request: {e}", file=sys.stderr)
        return None

In [ ]:
image_path = "/Users/chris/Desktop/gsframing.png"
prompt = """
You are an expert document analysis assistant. Analyze the invoice shown in the image and extract only the following fields:

- vendor_name
- invoice_number
- invoice_date
- total_amount

Return your answer as a valid, minified JSON object like this:
{"vendor_name": "...", "invoice_number": "...", "invoice_date": "...", "total_amount": ...}

Only use information visible in the image. If any field is missing or unreadable, use null.

Begin when ready. Do not include any additional text or explanations in your response.
"""


In [ ]:
image_path = "/Users/chris/Downloads/ccad.pdf"
prompt = """
You are an expert in extracting structured data from handwritten invoices.

Carefully examine the invoice shown in the image. Extract and return only the following fields:

- vendor_name
- invoice_number
- invoice_date (formatted as MM/DD/YYYY)
- total_amount (numeric or null)
- line_items: an array of objects, each with:
  - description
  - quantity (numeric or null)
  - unit_price (numeric or null)
  - line_total (numeric or null)

Use null for any fields that are missing or unreadable.

⚠️ Output only valid **minified JSON**.
❌ Do NOT include bullet points, markdown, or explanations.
❌ Do NOT prefix the response with “The invoice shows...” or any narrative.
🔒 Only return this format:
{"vendor_name":"...","invoice_number":"...","invoice_date":"...","total_amount":...,"line_items":[{"description":"...","quantity":...,"unit_price":...,"line_total":...}]}
"""

In [ ]:
#image_path = "/Users/chris/Desktop/gsframing.png"
image_path = "/Users/chris/Downloads/ccad.pdf"
ollama_32_vision_prompt = """
This is a bill from a vendor. Extract all visible text from the image exactly as it appears.
"""

In [ ]:
print(ollama_32_vision_prompt)

In [ ]:
resized_image_base64 = resize_convert_image_to_base64(image_path)

In [ ]:
image_data = base64.b64decode(resized_image_base64)
image = Image.open(io.BytesIO(image_data))
display(image)

In [ ]:
ollama_32_vision_prompt_response = get_ollama32_vision_response(
    image_base64=resized_image_base64,
    prompt=ollama_32_vision_prompt
)

In [ ]:
print(ollama_32_vision_prompt_response)

In [ ]:
ollama_32_text_prompt = f"""
You are a data extraction assistant. Interpret the following invoice text and extract only the structured data fields listed below.

### Extracted Text:
{ollama_32_vision_prompt_response}

### Required Output (strict JSON format only):

- vendor_name
- invoice_number
- invoice_date (formatted as MM/DD/YYYY)
- total_amount (numeric only, e.g. 10000.00)
- line_items: array of objects with:
  - description
  - quantity (numeric or null)
  - unit_price (numeric or null)
  - line_total (numeric or null)

Return only valid, minified JSON like this:
{{"vendor_name":"...","invoice_number":"...","invoice_date":"...","total_amount":...,"line_items":[{{"description":"...","quantity":...,"unit_price":...,"line_total":...}}]}}

You must respond with only valid minified JSON — no markdown, no commentary, no explanations, and no bullet points. If you do not know a value, use null. Do not include any natural language output. Only JSON is allowed in your response.
"""

In [ ]:
print(ollama_32_text_prompt)

In [ ]:
ollama_32_text_prompt_response = get_ollama32_text_response(
    prompt=ollama_32_text_prompt
)

In [ ]:
parsed_text_response = json.loads(ollama_32_text_prompt_response)
loaded_text_response = json.dumps(parsed_text_response, indent=2)
print(loaded_text_response)

In [ ]:
ollama_31_text_prompt = f"""
You are a data extraction assistant. Interpret the following invoice text and extract only the structured data fields listed below.

### Extracted Text:
{ollama_32_vision_prompt_response}

### Required Output (strict JSON format only):

- vendor_name
- invoice_number
- invoice_date (formatted as MM/DD/YYYY)
- total_amount (numeric only, e.g. 10000.00)
- line_items: array of objects with:
  - description
  - quantity (numeric or null)
  - unit_price (numeric or null)
  - line_total (numeric or null)

Return only valid, minified JSON like this:
{{"vendor_name":"...","invoice_number":"...","invoice_date":"...","total_amount":...,"line_items":[{{"description":"...","quantity":...,"unit_price":...,"line_total":...}}]}}

You must respond with only valid minified JSON — no markdown, no commentary, no explanations, and no bullet points. If you do not know a value, use null. Do not include any natural language output. Only JSON is allowed in your response.
"""

In [ ]:
print(ollama_31_text_prompt)

In [ ]:
text_response = get_ollama31_text_response(
    prompt=ollama_31_text_prompt
)

In [ ]:
print(type(text_response))

In [ ]:
import re

cleaned = text_response.strip()
# Remove triple backticks and any leading/trailing whitespace/newlines
cleaned = re.sub(r"^```[\s\n]*|[\s\n]*```$", "", cleaned)

parsed_text_response = json.loads(cleaned)
resp = json.dumps(parsed_text_response, indent=2)
print(resp)

In [ ]:
import jwt
token = jwt.encode(
    {"test": "value"},
    "secret",
    algorithm="HS256"
)
print(token)

In [ ]:
import jwt
print(jwt.__file__)
print(dir(jwt))

In [ ]:
OLLAMA_CHAT_URL = "http://localhost:11434/api/chat"

In [ ]:
MODEL = "llama3.2"

In [ ]:
def expanduser(path: str) -> str:
    return os.path.abspath(os.path.expanduser(path))

In [ ]:
def extract_pdf_text(path: str) -> str:
    reader = PdfReader(expanduser(path))
    return "\n\n".join((page.extract_text() or "") for page in reader.pages)

In [ ]:
def to_float(x):
    if x is None:
        return None
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x)
    s = re.sub(r"[^0-9.\-]", "", s)
    try:
        return float(s) if s else None
    except Exception:
        return None

In [ ]:
def sanity_check_totals(inv: dict) -> dict:
    items = inv.get("items") or []
    computed_subtotal = 0.0
    for it in items:
        qty = to_float(it.get("quantity"))
        unit = to_float(it.get("unit_price"))
        ext  = to_float(it.get("extended_price"))
        if ext is not None:
            computed_subtotal += ext
        elif qty is not None and unit is not None:
            computed_subtotal += qty * unit

    subtotal = to_float(inv.get("subtotal"))
    discount = to_float(inv.get("discount"))
    freight  = to_float(inv.get("freight"))
    tax      = to_float(inv.get("tax"))
    total    = to_float(inv.get("total_due"))

    sub_for_calc = subtotal if subtotal is not None else computed_subtotal
    expected = None
    if sub_for_calc is not None:
        expected = sub_for_calc - (discount or 0.0) + (freight or 0.0) + (tax or 0.0)

    return {
        "computed_subtotal_from_items": round(computed_subtotal, 2) if computed_subtotal is not None else None,
        "reported_subtotal": subtotal,
        "discount": discount,
        "freight": freight,
        "tax": tax,
        "expected_total": round(expected, 2) if expected is not None else None,
        "reported_total_due": total,
        "difference_total_minus_expected": round((total - expected), 2) if (expected is not None and total is not None) else None,
    }

In [ ]:
def call_ollama_json(content: str, schema_hint: str, temperature: float = 0.1, num_ctx: int = 8192) -> dict:
    """Call /api/chat with streaming OFF and parse strict JSON from the reply."""
    system_msg = (
        "You are an expert invoice parser. "
        "Return ONLY valid minified JSON. No markdown, no backticks, no commentary. "
        "Follow the requested schema and use null when unknown. Use numbers (no currency symbols). "
        "Dates must be ISO YYYY-MM-DD if present."
    )

    payload = {
        "model": MODEL,
        "stream": False,
        "options": {"temperature": temperature, "num_ctx": num_ctx},
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": schema_hint + "\n\n--- DOCUMENT START ---\n" + content + "\n--- DOCUMENT END ---"}
        ]
    }
    r = requests.post(OLLAMA_CHAT_URL, json=payload, timeout=300)
    r.raise_for_status()
    data = r.json()
    raw = (data.get("message") or {}).get("content") or ""

    # Try direct parse first
    try:
        return json.loads(raw)
    except Exception:
        pass

    # Fallback: grab the first {...} block
    m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if not m:
        raise ValueError("Model did not return JSON.\nRaw:\n" + raw)

    candidate = m.group(0)
    try:
        return json.loads(candidate)
    except Exception:
        # remove trailing commas, try again
        fixed = re.sub(r",\s*([}\]])", r"\1", candidate)
        return json.loads(fixed)

In [ ]:
schema = """
Extract the following fields as STRICT JSON:
{
  "vendor_name": string|null,
  "vendor_address": string|null,
  "bill_to": string|null,
  "ship_to": string|null,
  "invoice_number": string|null,
  "order_number": string|null,
  "po_number": string|null,
  "invoice_date": "YYYY-MM-DD"|null,
  "due_date": "YYYY-MM-DD"|null,
  "terms": string|null,
  "currency": "USD"|null,
  "items": [
    {
      "description": string|null,
      "sku": string|null,
      "quantity": number|null,
      "unit_price": number|null,
      "extended_price": number|null,
      "surcharges": number|null
    }
  ],
  "subtotal": number|null,
  "discount": number|null,
  "freight": number|null,
  "tax": number|null,
  "total_due": number|null,
  "notes": string|null
}
Rules:
- Use null if a field cannot be found exactly.
- If surcharges appear per-line, put them in item.surcharges; if global, use 'freight'.
- Do NOT invent data. Prefer values printed on the document.
- Return ONLY JSON.
"""

In [ ]:
pdf_path = "/Users/chris/Downloads/OHB - Associated Masonry Products - 0363361-IN - Mortar For Pavers - 55.0 - $240.44 - 3-21-2023.pdf"

In [ ]:
doc_text = extract_pdf_text(pdf_path)

In [ ]:
print(doc_text)

In [ ]:
MAX_CHARS = 24000
doc_for_llm = doc_text[:MAX_CHARS]

In [ ]:
extracted = call_ollama_json(doc_for_llm, schema)

In [ ]:
print("Parsed JSON:\n", json.dumps(extracted, indent=2))

In [ ]:
check = sanity_check_totals(extracted)
print("\nSanity check:\n", json.dumps(check, indent=2))

In [ ]:
def ask_llama(prompt, context_text):
    payload = {
        "model": "llama3.2",
        "stream": False,
        "options": {"temperature": 0.2, "num_ctx": 8192},
        "messages": [
            {"role": "system", "content": "You are a concise assistant. Summarize documents clearly."},
            {"role": "user", "content": f"{prompt}\n\n---\n{context_text}\n---"}
        ]
    }
    resp = requests.post(OLLAMA_URL, json=payload)
    return resp.json()["message"]["content"]

In [ ]:
answer = ask_llama("Summarize this document in 5 bullet points:", doc_text)
print(answer)

In [1]:
from azure.core.credentials import AzureKeyCredential

In [2]:
from azure.ai.documentintelligence import DocumentIntelligenceClient

In [3]:
import numpy as np

In [12]:
import json

In [4]:
ENDPOINT = "https://buildonedocintel.cognitiveservices.azure.com/"
KEY = "8QjcAGJmp5LIY6wmpJ7AqtkbxbetVqvQkFJcWnZkktv3hycLSCFdJQQJ99BFACYeBjFXJ3w3AAALACOGqPno"

In [5]:
def format_bounding_box(bounding_box):
    if not bounding_box:
        return "N/A"
    reshaped_bounding_box = np.array(bounding_box).reshape(-1, 2)
    return ", ".join(["[{}, {}]".format(x, y) for x, y in reshaped_bounding_box])

In [6]:
form_url = "/Users/chris/Downloads/OHB - Associated Masonry Products - 0363361-IN - Mortar For Pavers - 55.0 - $240.44 - 3-21-2023.pdf"

In [7]:
document_intelligence_client  = DocumentIntelligenceClient(
    endpoint=ENDPOINT, credential=AzureKeyCredential(KEY)
)

In [8]:
with open(form_url, "rb") as f:
    poller = document_intelligence_client.begin_analyze_document(
        "prebuilt-read",
        body=f,
        content_type="application/pdf"
    )

In [9]:
result = poller.result()

In [10]:
print ("Document contains content: ", result.content)

Document contains content:  Page:
1
Invoice
Associated Masonry Products, Inc. 450 Allied Drive Nashville, TN 37211 (615) 331-1000
Invoice Number: 0363361-IN Invoice Date: 3/21/2023
Order Number: 0287007
Order Date 3/16/2023
Salesperson: 0010
Customer Number: 01-0003572
Customer PO No.
Sold To:
Rogers Build LLC
PO Box 594
Brentwood, TN 37024
Ship To:
Rogers Build LLC
1809 OLD HICKORY BLVD
Brentwood, TN 37024
Confirm To:
Fax:
Job No.
Ship VIA
F.O.B.
Terms
1809OLDH
Net 30 days
Item Number
Unit
Ordered
Shipped
Back Ordered
Price
Amount
NAT-C-IVORY-BUF COOSA IVORY BUFF "N"
BAG
7.00
7.00
0.00
29.6595
207.62
/01-TRANSPORT 01 Transportation Surcharge
Whse: 01A
12.46
All cash sales are FINAL!
Net Invoice:
220.08
Less Discount:
0.00
Freight:
0.00
Sales Tax:
20.36
Invoice Total:
240.44


In [13]:
print(json.dumps(result.as_dict(), indent=2))

{
  "apiVersion": "2024-11-30",
  "modelId": "prebuilt-read",
  "stringIndexType": "textElements",
  "content": "Page:\n1\nInvoice\nAssociated Masonry Products, Inc. 450 Allied Drive Nashville, TN 37211 (615) 331-1000\nInvoice Number: 0363361-IN Invoice Date: 3/21/2023\nOrder Number: 0287007\nOrder Date 3/16/2023\nSalesperson: 0010\nCustomer Number: 01-0003572\nCustomer PO No.\nSold To:\nRogers Build LLC\nPO Box 594\nBrentwood, TN 37024\nShip To:\nRogers Build LLC\n1809 OLD HICKORY BLVD\nBrentwood, TN 37024\nConfirm To:\nFax:\nJob No.\nShip VIA\nF.O.B.\nTerms\n1809OLDH\nNet 30 days\nItem Number\nUnit\nOrdered\nShipped\nBack Ordered\nPrice\nAmount\nNAT-C-IVORY-BUF COOSA IVORY BUFF \"N\"\nBAG\n7.00\n7.00\n0.00\n29.6595\n207.62\n/01-TRANSPORT 01 Transportation Surcharge\nWhse: 01A\n12.46\nAll cash sales are FINAL!\nNet Invoice:\n220.08\nLess Discount:\n0.00\nFreight:\n0.00\nSales Tax:\n20.36\nInvoice Total:\n240.44",
  "pages": [
    {
      "pageNumber": 1,
      "angle": 0,
      "width

In [14]:
for idx, style in enumerate(result.styles):
    print(
        "Document contains {} content".format(
            "handwritten" if style.is_handwritten else "no handwritten"
        )
    )

In [15]:
for page in result.pages:
    print("----Analyzing Read from page #{}----".format(page.page_number))
    print(
        "Page has width: {} and height: {}, measured with unit: {}".format(
            page.width, page.height, page.unit
        )
    )

    for line_idx, line in enumerate(page.lines):
        print(
            "...Line # {} has text content '{}' within bounding box '{}'".format(
                line_idx,
                line.content,
                format_bounding_box(line.polygon),
            )
        )

    for word in page.words:
        print(
            "...Word '{}' has a confidence of {}".format(
                word.content, word.confidence
            )
        )

print("----------------------------------------")

----Analyzing Read from page #1----
Page has width: 8.5 and height: 11.0, measured with unit: inch
...Line # 0 has text content 'Page:' within bounding box '[7.4351, 0.2578], [7.7551, 0.2626], [7.7551, 0.3772], [7.4351, 0.3772]'
...Line # 1 has text content '1' within bounding box '[8.1466, 0.2674], [8.1992, 0.2674], [8.1992, 0.3533], [8.1371, 0.3533]'
...Line # 2 has text content 'Invoice' within bounding box '[3.9157, 0.4297], [4.4983, 0.4345], [4.4983, 0.5872], [3.9157, 0.5825]'
...Line # 3 has text content 'Associated Masonry Products, Inc.' within bounding box '[0.5683, 0.9214], [2.2969, 0.9214], [2.2969, 1.0503], [0.5683, 1.0456]'
...Line # 4 has text content '450 Allied Drive' within bounding box '[0.573, 1.0503], [1.3562, 1.0503], [1.3562, 1.1649], [0.573, 1.1602]'
...Line # 5 has text content 'Nashville, TN 37211' within bounding box '[0.573, 1.1793], [1.5663, 1.1745], [1.5663, 1.2891], [0.573, 1.2938]'
...Line # 6 has text content '(615) 331-1000' within bounding box '[0.5778

In [ ]:
import time

In [ ]:



model = "prebuilt-invoice"
api_version = "2024-11-30"  # check latest in docs

In [ ]:
with open(os.path.expanduser(file_path), "rb") as f:
    url = f"{ENDPOINT}documentintelligence/documentModels/{model}:analyze?_overload=analyzeDocument&api-version={api_version}"
    headers = {
        "Ocp-Apim-Subscription-Key": KEY,
        "Content-Type": "application/json"  # or image/jpeg, image/png
    }
    resp = requests.post(url, headers=headers, data=f)

if resp.status_code != 202:
    raise RuntimeError(f"Error {resp.status_code}: {resp.text}")

# poll the operation-location
op_url = resp.headers["operation-location"]

In [ ]:
while True:
    poll = requests.get(op_url, headers={"Ocp-Apim-Subscription-Key": KEY})
    result = poll.json()
    status = result.get("status")
    if status in ("succeeded", "failed"):
        break
    time.sleep(2)

if status != "succeeded":
    raise RuntimeError(f"Analysis failed: {json.dumps(result, indent=2)}")

# Full raw result
print(json.dumps(result, indent=2))